In [ ]:
import nltk
import pandas as pd
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag

nltk.download('punkt')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')

# Load dataset
corpus = pd.read_csv("Twitter_Data.csv")
corpus = corpus.dropna(subset=['clean_text', 'category'])

print(corpus.head())
print(corpus.shape)
print(corpus['category'].value_counts())

# POS tag mapping for lemmatization
POS_TAG_MAP = {
    'N': 'n',
    'V': 'v',
    'R': 'r',
    'J': 'a'
}

def get_wordnet_pos(tag):
    return POS_TAG_MAP.get(tag[0], 'n')

lemmatizer = WordNetLemmatizer()

In [ ]:
import nltk
import re
import numpy as np
import pandas as pd
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from scipy.sparse import hstack
import gensim.downloader as api

# Load lightweight Word2Vec model
print("Loading Word2Vec model...")
w2v_model = api.load("glove-twitter-25")
print("Done.")

# Load dataset
corpus = pd.read_csv("Twitter_Data.csv")
corpus = corpus.dropna(subset=['clean_text', 'category'])
corpus = corpus.sample(n=1000, random_state=50)

# POS tag mapping
POS_TAG_MAP = {'N': 'n', 'V': 'v', 'R': 'r', 'J': 'a'}

def get_wordnet_pos(tag):
    return POS_TAG_MAP.get(tag[0], 'n')

# Preprocessing function
def pretraitement(text):
    text = re.sub(r'[^\w\s]', '', str(text))
    tokens = word_tokenize(text.lower())
    lemmatizer = WordNetLemmatizer()
    lemmas, pos_tags = zip(*[
        (lemmatizer.lemmatize(word, get_wordnet_pos(tag)), tag)
        for word, tag in nltk.pos_tag(tokens)
    ]) if tokens else ([], [])
    return pd.Series([' '.join(lemmas), ' '.join(pos_tags)])

print("Preprocessing...")
corpus[['lemma', 'pos']] = corpus['clean_text'].apply(pretraitement)
print("Done.")

# POS one-hot encoding
mlb = MultiLabelBinarizer()
pos_one_hot = mlb.fit_transform(corpus['pos'].apply(lambda x: x.split()))

# TF-IDF vectorization
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus['lemma'])

# Word2Vec vectorization
word2vec_vectors = []
for phrase in corpus['lemma'].str.split():
    vectors = [w2v_model[word] for word in phrase if word in w2v_model]
    if vectors:
        word2vec_vectors.append(np.mean(vectors, axis=0))
    else:
        word2vec_vectors.append(np.zeros(w2v_model.vector_size))

# ── Training ──────────────────────────────────────────────

model_reg = LogisticRegression(max_iter=1000)

# TF-IDF
X_train1, X_test1, y_train1, y_test1 = train_test_split(
    tfidf_matrix, corpus['category'], test_size=0.2, random_state=42)
model_reg.fit(X_train1, y_train1)
y_pred_tfidf = model_reg.predict(X_test1)
print("Accuracy TF-IDF:", accuracy_score(y_test1, y_pred_tfidf))
print(classification_report(y_test1, y_pred_tfidf))

# Word2Vec
X_train2, X_test2, y_train2, y_test2 = train_test_split(
    word2vec_vectors, corpus['category'], test_size=0.2, random_state=42)
model_reg.fit(X_train2, y_train2)
y_pred_w2v = model_reg.predict(X_test2)
print("Accuracy Word2Vec:", accuracy_score(y_test2, y_pred_w2v))
print(classification_report(y_test2, y_pred_w2v))

# TF-IDF + POS
X_combined = hstack([tfidf_matrix, pos_one_hot])
X_train3, X_test3, y_train3, y_test3 = train_test_split(
    X_combined, corpus['category'], test_size=0.2, random_state=42)
model_reg.fit(X_train3, y_train3)
y_pred_tfidf_pos = model_reg.predict(X_test3)
print("Accuracy TF-IDF + POS:", accuracy_score(y_test3, y_pred_tfidf_pos))
print(classification_report(y_test3, y_pred_tfidf_pos))